# B2B SaaS Attribution Walkthrough

This notebook mirrors the client-facing case study in the repository. It uses the reusable package code and the same recommendation logic that drives the exported executive summary.

## Buyer Problem

A B2B SaaS team can have healthy pipeline creation and still make poor budget decisions if it relies on a closing-touch reporting model. The goal here is to show, with synthetic public data, how a more defensible attribution workflow changes the budget conversation.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from marketing_attribution.models import build_attribution_report, load_touchpoints
from marketing_attribution.reporting import build_executive_summary, build_recommendation_table

df = load_touchpoints(ROOT / 'data' / 'touchpoints.csv')
report = build_attribution_report(df)
report['recommendations'] = build_recommendation_table(report['channel_scorecard'])
df.head()

,journey_id,account_id,segment,region,industry,channel,funnel_stage,touch_position,total_touches,touch_timestamp,is_paid_channel,cost,converted,conversion_timestamp,days_to_conversion,revenue
0,1,1004,Mid-Market,North America,Healthcare,linkedin_ads,awareness,1,4,2024-07-12 09:00:00,1,78.69,0,NaT,NaN,0.0
1,1,1004,Mid-Market,North America,Healthcare,google_search,consideration,2,4,2024-07-13 09:00:00,1,51.69,0,NaT,NaN,0.0
2,1,1004,Mid-Market,North America,Healthcare,email_nurture,decision,3,4,2024-08-03 10:00:00,1,14.43,0,NaT,NaN,0.0
3,1,1004,Mid-Market,North America,Healthcare,partner_referral,decision,4,4,2024-08-10 12:00:00,1,271.76,0,NaT,NaN,0.0
4,2,637,SMB,North America,Healthcare,linkedin_ads,awareness,1,4,2024-04-07 14:00:00,1,88.30,0,NaT,NaN,0.0


In [2]:
converted = df[df['converted'] == 1]
kpis = {
    'accounts': int(df['account_id'].nunique()),
    'journeys': int(df['journey_id'].nunique()),
    'converting_journeys': int(converted['journey_id'].nunique()),
    'conversion_rate': round(converted['journey_id'].nunique() / df['journey_id'].nunique(), 4),
    'pipeline': round(converted.groupby('journey_id')['revenue'].max().sum(), 2),
    'median_touches': float(converted.groupby('journey_id')['total_touches'].max().median()),
}
kpis

{'accounts': 1942,
 'journeys': 4500,
 'converting_journeys': 1205,
 'conversion_rate': 0.2678,
 'pipeline': np.float64(54973197.61),
 'median_touches': 6.0}

## Channel Revenue by Model

In [3]:
report['revenue_pivot'].round(0)

model,last_touch,linear,time_decay,u_shaped
channel,,,,
retargeting,14562077.0,6581736.0,11065075.0,6980923.0
email_nurture,12122896.0,6382553.0,9948743.0,6039772.0
partner_referral,14275599.0,5781997.0,9633184.0,6645844.0
direct,14012626.0,5534185.0,9143218.0,6534336.0
google_search,0.0,6917964.0,4214089.0,2352756.0
webinar,0.0,6392268.0,3845662.0,2065641.0
review_sites,0.0,5288702.0,3419127.0,1712848.0
organic_search,0.0,5782402.0,1904756.0,10257461.0
linkedin_ads,0.0,6311391.0,1799344.0,12383617.0


## Top Converting Paths

In [4]:
report['top_paths'].head(10)

,path,journeys,revenue
0,organic_search > google_search > retargeting,8,458136.74
1,linkedin_ads > google_search > email_nurture,8,327905.05
2,linkedin_ads > google_search > retargeting,7,210140.95
3,linkedin_ads > google_search > partner_referral,6,585848.74
4,organic_search > review_sites > retargeting,6,427377.96
5,organic_search > google_search > direct,5,347983.80
6,linkedin_ads > google_search > direct,5,242407.21
7,organic_search > linkedin_ads > webinar > goog...,5,158429.74
8,linkedin_ads > webinar > google_search > revie...,5,79311.30
9,linkedin_ads > google_search > webinar > email...,4,490647.03


## Recommendation Scorecard

In [5]:
report['recommendations'][['channel', 'primary_recommendation', 'rationale', 'time_decay_roas', 'time_decay_uplift']].round(2)

,channel,primary_recommendation,rationale,time_decay_roas,time_decay_uplift
0,google_search,invest,"under-credited by last-touch, strong time-deca...",20.75,4214088.76
1,review_sites,monitor,"under-credited by last-touch, strong time-deca...",29.38,3419127.12
2,retargeting,monitor,"over-credited by last-touch, strong time-decay...",106.28,-3497002.30
3,email_nurture,monitor,"over-credited by last-touch, strong time-decay...",337.22,-2174152.96
4,webinar,monitor,"under-credited by last-touch, weaker time-deca...",6.82,3845661.63
5,linkedin_ads,monitor,"under-credited by last-touch, weaker time-deca...",7.45,1799344.31
6,partner_referral,trim,"over-credited by last-touch, less efficient th...",17.52,-4642414.92


## What this would look like in a real engagement

In production, the same workflow would connect campaign touches, lifecycle timestamps, and CRM opportunity history. The public version here uses synthetic data so the methodology can be reviewed openly.

## Executive Summary Draft

In [6]:
print(build_executive_summary(df, report))

# Executive Summary

_Generated 2026-03-30 19:36 UTC_

## Why this case study matters

This public demo shows the kind of attribution engagement a B2B SaaS CMO would buy:
a unified touchpoint model, defensible attribution logic, executive-ready visuals, and budget recommendations tied to pipeline outcomes.

## Dataset snapshot

- Accounts represented: 1,942
- Buying journeys analyzed: 4,500
- Converting journeys: 1,205 (26.8% conversion rate)
- Attributed pipeline in the sample: $54,973,198
- Average won opportunity value: $45,621
- Median days from first touch to conversion: 57.1
- Median touches in winning journeys: 6

## What the analysis shows

- Last-touch reporting still assigns $54,973,198 of pipeline, but it materially under-values assist channels earlier in the buying journey.
- Winning journeys usually require 6 touches, which makes single-touch reporting incomplete for budget allocation.
- Early demand creation is led by `linkedin_ads` and `organic_search`, while conversion 